# D11 Calculation Pathway Effects

Analyze how `D11` changes across calculation pathways while holding the observed ECTP slab fixed. 

In [1]:
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from notebook_utils import (
    create_ectp_slabs,
    extract_d11,
    finite_percentile,
    finite_positive,
    hess_rcparams,
    load_pits,
    pathway_key,
    prepare_spread_rank_table,
    slab_attribute_record,
)
from paper_figure_utils import (
    build_d11_paired_pathway_effects_figure,
    build_d11_spread_attribute_correlations_figure,
    format_method_path,
    notebook_latex,
    save_paper_figure,
)
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig
from snowpyt_mechparams.graph import default_graph
from snowpyt_mechparams.pathway import find_parameterizations

hess_rcparams()

TOP_N = 8

## Load Data and Enumerate D11 Pathways

Use the same ECTP slab definition as the slab-weight input analysis: Slab is the set of layers above the ECTP layer of propagation

In [2]:
pits = load_pits()
ectp_slabs = create_ectp_slabs(pits)
total_slabs = len(ectp_slabs)

pathways = find_parameterizations(default_graph, default_graph.get_node('D11'))
engine = ExecutionEngine()
config = ExecutionConfig(include_method_uncertainty=False) # input uncertainty only

print(f'Loaded {len(pits):,} pits and {total_slabs:,} ECTP slabs')
print(f'D11 pathways: {len(pathways)}')

Loaded 50,278 pits and 14,776 ECTP slabs
D11 pathways: 32


## Execute D11 Pathways

Store one row per slab-pathway attempt. Failed attempts remain in `d11_results` with `NaN` values so coverage can be computed directly from the same table.

In [3]:
records = []

for slab_index, slab in enumerate(ectp_slabs):
    slab_attrs = slab_attribute_record(slab_index, slab)
    results = engine.execute_all(slab, 'D11', config=config, pathways=pathways)

    for pathway_result in results.pathways.values():
        methods = pathway_result.methods_used
        d11_nominal, d11_std = extract_d11(pathway_result)
        d11_success = finite_positive(d11_nominal)

        records.append(
            {
                **slab_attrs,
                'pathway': pathway_key(methods),
                'pathway_description': pathway_result.pathway_description,
                'density_method': methods.get('density', 'data_flow'),
                'emod_method': methods.get('elastic_modulus', 'unknown'),
                'nu_method': methods.get('poissons_ratio', 'unknown'),
                'pathway_result_success': pathway_result.success,
                'success': d11_success,
                'D11_nominal': d11_nominal if d11_success else math.nan,
                'D11_std': d11_std if d11_success else math.nan,
                'D11_rel_uncertainty': d11_std / abs(d11_nominal) if d11_success else math.nan,
            }
        )

d11_results = pd.DataFrame(records)
d11_success = d11_results[d11_results['success']].copy()

print(f'Pathway attempts: {len(d11_results):,}')
print(f'Successful D11 calculations: {len(d11_success):,}')
print(f'Slabs with at least one successful D11 value: {d11_success["slab_index"].nunique():,}')
#d11_results.head()

Pathway attempts: 472,832
Successful D11 calculations: 8,858
Slabs with at least one successful D11 value: 823


## Top 8 Coverage Pathways

Rank pathways by successful ECTP slab coverage. 

In [4]:
pathway_coverage = (
    d11_results.groupby(['pathway', 'density_method', 'emod_method', 'nu_method'])
    .agg(
        successful_slabs=('success', 'sum'),
        attempted_slabs=('success', 'count'),
    )
    .reset_index()
    .sort_values(['successful_slabs', 'pathway'], ascending=[False, True])
)
pathway_coverage['coverage_percent'] = 100.0 * pathway_coverage['successful_slabs'] / total_slabs

top_pathways = pathway_coverage.head(TOP_N)['pathway'].tolist()
coverage_table = pd.DataFrame(
    {
        'Pathway': [format_method_path(*pathway.split(' -> ')) for pathway in top_pathways],
        'Successful slabs': pathway_coverage.head(TOP_N)['successful_slabs'].map(lambda value: f'{int(value):,}'),
        'Coverage (%)': pathway_coverage.head(TOP_N)['coverage_percent'].map(lambda value: f'{value:.1f}'),
    }
)

coverage_table

# Update index to be ranking
coverage_table.index = coverage_table.index.get_indexer(coverage_table.index)
coverage_table.index.name = 'Rank'
coverage_table


,Pathway,Successful slabs,Coverage (%)
Rank,,,
0,Kim and Jamieson Table 2 -> Schottner et al. (...,743,5.0
1,Kim and Jamieson Table 2 -> Wautier et al. (20...,743,5.0
2,Geldsetzer and Jamieson (2000) -> Schottner et...,740,5.0
3,Geldsetzer and Jamieson (2000) -> Wautier et a...,740,5.0
4,Geldsetzer and Jamieson (2000) -> Kochle and S...,733,5.0
5,Kim and Jamieson Table 2 -> Kochle and Schneeb...,634,4.3
6,Kim and Jamieson Table 2 -> Bergfeld et al. (2...,498,3.4
7,Geldsetzer and Jamieson (2000) -> Bergfeld et ...,475,3.2


## Top-8 Slab Subset

Keep only slabs where all top-8 pathways produced finite, positive D11 values.

In [5]:
top8_success = d11_success[d11_success['pathway'].isin(top_pathways)].copy()
common_counts = top8_success.groupby('slab_index')['pathway'].nunique()
common_slab_indices = common_counts[common_counts == TOP_N].index

d11_common = top8_success[top8_success['slab_index'].isin(common_slab_indices)].copy()
d11_common['pathway'] = pd.Categorical(d11_common['pathway'], categories=top_pathways, ordered=True)
d11_common = d11_common.sort_values(['pathway', 'slab_index'])

d11_wide = d11_common.pivot(
    index=['slab_index', 'slab_id', 'pit_id'],
    columns='pathway',
    values='D11_nominal',
).reindex(columns=top_pathways)

print(f'Top-8 pathways: {TOP_N}')
print(f'Slabs successful for all top-8 pathways: {len(common_slab_indices):,}')

d11_wide.head()


Top-8 pathways: 8
Slabs successful for all top-8 pathways: 412


,,pathway,kim_jamieson_table2 -> schottner -> kochle,kim_jamieson_table2 -> wautier -> kochle,geldsetzer -> schottner -> kochle,geldsetzer -> wautier -> kochle,geldsetzer -> kochle -> kochle,kim_jamieson_table2 -> kochle -> kochle,kim_jamieson_table2 -> bergfeld -> kochle,geldsetzer -> bergfeld -> kochle
slab_index,slab_id,pit_id,,,,,,,,
98,1036_slab_0,1036,1.033784e+05,3.897811e+06,9.853946e+04,3.803908e+06,1.058172e+06,1.171239e+06,2.183333e+05,2.085478e+05
141,1057_slab_0,1057,3.364213e+07,1.037104e+09,3.518988e+07,1.061108e+09,2.822266e+08,2.740997e+08,6.979665e+07,7.286510e+07
142,1057_slab_1,1057,3.364213e+07,1.037104e+09,3.518988e+07,1.061108e+09,2.822266e+08,2.740997e+08,6.979665e+07,7.286510e+07
143,1057_slab_2,1057,3.364213e+07,1.037104e+09,3.518988e+07,1.061108e+09,2.822266e+08,2.740997e+08,6.979665e+07,7.286510e+07
144,1057_slab_3,1057,3.364213e+07,1.037104e+09,3.518988e+07,1.061108e+09,2.822266e+08,2.740997e+08,6.979665e+07,7.286510e+07


## Paired Pathway Spread

Summarize how far pathways diverge within each slab. The primary spread metric is `log10(max/min)`, which is unitless and symmetric on a ratio scale; absolute D11 range is retained only as supporting context.

In [9]:
min_pathways = d11_wide.idxmin(axis=1).astype(str)
max_pathways = d11_wide.idxmax(axis=1).astype(str)
slab_attr = d11_common.groupby(['slab_index', 'slab_id', 'pit_id']).agg(
    n_layers=('n_layers', 'first'),
    total_thickness_cm=('total_thickness_cm', 'first'),
    slope_angle_deg=('slope_angle_deg', 'first'),
    layer_thickness_mean_cm=('layer_thickness_mean_cm', 'first'),
    layer_thickness_std_cm=('layer_thickness_std_cm', 'first'),
    layer_thickness_cv=('layer_thickness_cv', 'first'),
    layer_thickness_max_cm=('layer_thickness_max_cm', 'first'),
    hand_hardness_index_mean=('hand_hardness_index_mean', 'first'),
    hand_hardness_index_std=('hand_hardness_index_std', 'first'),
    hand_hardness_index_range=('hand_hardness_index_range', 'first'),
    hand_hardness_index_cv=('hand_hardness_index_cv', 'first'),
    hand_hardness_index_weighted_mean=('hand_hardness_index_weighted_mean', 'first'),
).reset_index()

paired_spread = pd.DataFrame(
    {
        'slab_index': [idx[0] for idx in d11_wide.index],
        'slab_id': [idx[1] for idx in d11_wide.index],
        'pit_id': [idx[2] for idx in d11_wide.index],
        'min_D11': d11_wide.min(axis=1).to_numpy(),
        'max_D11': d11_wide.max(axis=1).to_numpy(),
        'min_pathway': min_pathways.to_numpy(),
        'max_pathway': max_pathways.to_numpy(),
    }
).merge(slab_attr, on=['slab_index', 'slab_id', 'pit_id'], how='left')

paired_spread.head()

,slab_index,slab_id,pit_id,min_D11,max_D11,min_pathway,max_pathway,n_layers,total_thickness_cm,slope_angle_deg,layer_thickness_mean_cm,layer_thickness_std_cm,layer_thickness_cv,layer_thickness_max_cm,hand_hardness_index_mean,hand_hardness_index_std,hand_hardness_index_range,hand_hardness_index_cv,hand_hardness_index_weighted_mean
0,98,1036_slab_0,1036,9.853946e+04,3.897811e+06,geldsetzer -> schottner -> kochle,kim_jamieson_table2 -> wautier -> kochle,1,5.0,30.0,5.0,0.0,0.0,5.0,3.67,0.0,0.0,0.0,3.67
1,141,1057_slab_0,1057,3.364213e+07,1.061108e+09,kim_jamieson_table2 -> schottner -> kochle,geldsetzer -> wautier -> kochle,1,30.0,37.0,30.0,0.0,0.0,30.0,4.00,0.0,0.0,0.0,4.00
2,142,1057_slab_1,1057,3.364213e+07,1.061108e+09,kim_jamieson_table2 -> schottner -> kochle,geldsetzer -> wautier -> kochle,1,30.0,37.0,30.0,0.0,0.0,30.0,4.00,0.0,0.0,0.0,4.00
3,143,1057_slab_2,1057,3.364213e+07,1.061108e+09,kim_jamieson_table2 -> schottner -> kochle,geldsetzer -> wautier -> kochle,1,30.0,37.0,30.0,0.0,0.0,30.0,4.00,0.0,0.0,0.0,4.00
4,144,1057_slab_3,1057,3.364213e+07,1.061108e+09,kim_jamieson_table2 -> schottner -> kochle,geldsetzer -> wautier -> kochle,1,30.0,37.0,30.0,0.0,0.0,30.0,4.00,0.0,0.0,0.0,4.00


## Smallest and Widest Paired Spread Slabs

Rank filtered slabs by `log10(max/min)`. Absolute D11 range is included to preserve physical scale, but the ratio-based spread is the paired comparison metric.

In [ ]:
smallest_spread = paired_spread_filtered.sort_values('log10_pathway_spread', ascending=True).head(TOP_N)
widest_spread = paired_spread_filtered.sort_values('log10_pathway_spread', ascending=False).head(TOP_N)
smallest_spread_table = prepare_spread_rank_table(
    smallest_spread,
    top_n=TOP_N,
    pathway_formatter=lambda value: format_method_path(*value.split(' -> '), short=True),
)
widest_spread_table = prepare_spread_rank_table(
    widest_spread,
    top_n=TOP_N,
    pathway_formatter=lambda value: format_method_path(*value.split(' -> '), short=True),
)

print('Smallest filtered paired spread slabs:')
display(smallest_spread_table)
print('Widest filtered paired spread slabs:')
widest_spread_table

## Attribute Correlations With Paired Pathway Spread

Use Spearman rank correlations to identify slab attributes that align most strongly with paired pathway spread, measured as `log10(max D11 / min D11)` within the same slab.

In [ ]:
attribute_labels = {
    'n_layers': 'Number of layers',
    'total_thickness_cm': 'Total slab thickness',
    'layer_thickness_mean_cm': 'Mean layer thickness'
}

correlation_rows = []
for column, label in attribute_labels.items():
    subset = paired_spread_filtered[[column, 'log10_pathway_spread']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(subset) < 3 or subset[column].nunique() < 2:
        continue
    result = spearmanr(subset[column], subset['log10_pathway_spread'])
    if not math.isfinite(result.statistic):
        continue
    correlation_rows.append(
        {
            'Attribute': label,
            'Spearman rho': float(result.statistic),
            'Abs Spearman rho': abs(float(result.statistic)),
            'p value': float(result.pvalue),
            'n': len(subset),
        }
    )

attribute_correlations = (
    pd.DataFrame(correlation_rows)
    .sort_values(['Abs Spearman rho', 'Attribute'], ascending=[False, True])
    .reset_index(drop=True)
)

attribute_correlation_table = pd.DataFrame(
    {
        'Attribute': attribute_correlations['Attribute'],
        'Spearman rho': attribute_correlations['Spearman rho'].map(lambda value: f'{value:.2f}'),
        'p value': attribute_correlations['p value'].map(lambda value: '<0.001' if value < 0.001 else f'{value:.3f}'),
        'n': attribute_correlations['n'].map(lambda value: f'{int(value):,}'),
    }
)

correlation_figure = build_d11_spread_attribute_correlations_figure(
    attribute_correlations,
    top_n=10,
    xlabel=r'Spearman $\rho$ with $\log_{10}(D_{11,max}/D_{11,min})$',
)
correlation_paths = save_paper_figure(
    correlation_figure,
    'd11_paired_spread_attribute_correlations',
)
display(correlation_figure)

print('Correlation figure exports:', correlation_paths)
attribute_correlation_table.head(10)

## Copy-Ready LaTeX Tables

In [ ]:
print('Top-8 pathway coverage table:')
print(notebook_latex(coverage_table))
print()
print('Paired pathway effects relative to within-slab median:')
print(notebook_latex(paired_effect_table))
print()
print('Smallest filtered paired spread slabs:')
print(notebook_latex(smallest_spread_table))
print()
print('Widest filtered paired spread slabs:')
print(notebook_latex(widest_spread_table))
print()
print('Attribute correlations with paired pathway spread:')
print(notebook_latex(attribute_correlation_table.head(10)))